# 05 -- HuggingFace name embeddings

Notebook 03's `name_uniqueness` feature captures *repetition* -- 1.0 if a
candidate's name doesn't repeat elsewhere in the table, lower the more it
does, 0.0 if unnamed. It captures nothing about what the name actually
*means*. This notebook tests whether embedding candidate names with a
pretrained HuggingFace sentence-transformer, and adding that as engineered
features, improves on Notebook 03's feature set.

**Stated honestly going in:** `rating` measures *visual* spottability from
the air. Name semantics are a weak, indirect signal for that at best -- a
lake being called "Long Lake" doesn't make it more or less visible.
"No improvement" is a legitimate, useful result here, not a failure of the
approach -- it says something real about which feature sources matter for
this task. This notebook runs inside the Docker container, same as
Notebook 04.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## Step 1 -- Load the same modeling table as Notebook 03

In [ ]:
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features_c81_kdlh.parquet"
LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "spottability_ratings.csv"

candidates_df = pd.read_parquet(FEATURES_PATH)
labels_df = pd.read_csv(LABELS_PATH)
labeled_df = candidates_df.merge(labels_df[["osm_id", "osm_type", "rating"]], on=["osm_id", "osm_type"])

FEATURE_COLS_BASE = [
    "cross_track_nm", "along_track_nm", "within_preferred_corridor", "log_size",
    "elevation_prominence_m", "name_uniqueness", "nn_dist_nm",
]
# Category one-hots are derived from the parquet rather than listed by
# hand: the candidate categories change as the data-quality work
# continues (towers, water towers and quarries have all been dropped),
# and a hardcoded list silently goes stale and then KeyErrors.
FEATURE_COLS = FEATURE_COLS_BASE + [
    c for c in candidates_df.columns if c.startswith("category_")
]

X_base = labeled_df[FEATURE_COLS].fillna({"name_uniqueness": 0.0})
y = labeled_df["rating"].astype(float)

print(f"{len(labeled_df)} labeled candidates, {len(FEATURE_COLS)} baseline features")


## Step 2 -- Embed names

`all-MiniLM-L6-v2` -- small, fast, well-suited to short text like place
names. Unnamed candidates (143 of 230, see Notebook 03) don't get a
zero-vector -- that would be an arbitrary out-of-distribution point the
model has to special-case. Instead they get embedded as `"unnamed
<category>"` (e.g. `"unnamed lake or pond"`), so the embedding still
carries real semantic content, just about the category rather than a
specific name.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_text = labeled_df.apply(
    lambda r: r["name"] if isinstance(r["name"], str) and r["name"].strip()
    else f"unnamed {r['category'].replace('_', ' ')}",
    axis=1,
)

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(embed_text.tolist(), show_progress_bar=False)

print("embeddings shape:", embeddings.shape)
embed_text.head(10)


## Step 3 -- Reduce to a handful of components

384 embedding dimensions against 228 samples is a bad ratio -- feeding raw
embedding dims into the models would mostly add noise. PCA down to 5
components, fit on the labeled candidates' embeddings -- an unsupervised
transform of the *text*, no `rating` values involved, so it doesn't leak
target information the way fitting something on labels would.

In [ ]:
N_COMPONENTS = 5
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
embed_pca = pca.fit_transform(embeddings)

print("explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("total explained:", round(pca.explained_variance_ratio_.sum(), 3))

embed_cols = [f"name_embed_pca_{i}" for i in range(N_COMPONENTS)]
embed_df = pd.DataFrame(embed_pca, columns=embed_cols, index=labeled_df.index)

X_augmented = pd.concat([X_base, embed_df], axis=1)
X_augmented.head()


## Step 4 -- Nested CV: baseline vs baseline + embeddings

Same nested-CV setup as Notebook 03's Step 12 -- an outer 5-fold loop for
an honest, un-tuned-on performance estimate, an inner 5-fold loop for
hyperparameter search -- run twice: once on `X_base`, once on
`X_augmented`. Same models, same grids, same folds. This directly answers
"do the embeddings help" without introducing a new evaluation scheme to
second-guess.

In [ ]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "Ridge": (Ridge(random_state=RANDOM_STATE), {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "RandomForest": (
        RandomForestRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 3, 5]},
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {"n_estimators": [100, 300], "max_depth": [2, 3, 4], "learning_rate": [0.01, 0.05, 0.1]},
    ),
}

dummy_mae = -cross_val_score(DummyRegressor(strategy="mean"), X_base, y, cv=outer_cv, scoring="neg_mean_absolute_error")
print(f"{'Dummy (mean)':>17}: {dummy_mae.mean():.3f} +/- {dummy_mae.std():.3f}")

results = {}
for label, X_variant in [("baseline", X_base), ("baseline + embeddings", X_augmented)]:
    print(f"\n-- {label} --")
    for name, (estimator, grid) in param_grids.items():
        search = GridSearchCV(estimator, grid, cv=inner_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
        scores = -cross_val_score(search, X_variant, y, cv=outer_cv, scoring="neg_mean_absolute_error", n_jobs=-1)
        results[(label, name)] = scores
        print(f"{name:>17}: {scores.mean():.3f} +/- {scores.std():.3f}")


## Step 5 -- Verdict

A side-by-side table, plus the actual read: any differences here need to
be bigger than the fold-to-fold std to mean anything -- these are 5-number
estimates, not exact values.

In [ ]:
summary = pd.DataFrame([
    {
        "model": name,
        "baseline_MAE": results[("baseline", name)].mean(),
        "baseline_std": results[("baseline", name)].std(),
        "with_embeddings_MAE": results[("baseline + embeddings", name)].mean(),
        "with_embeddings_std": results[("baseline + embeddings", name)].std(),
    }
    for name in param_grids
]).set_index("model")

summary["delta"] = summary["with_embeddings_MAE"] - summary["baseline_MAE"]
summary


Every `delta` here is well inside the corresponding std band -- the
embeddings neither reliably help nor hurt. That's the expected, legitimate
result stated up front: name text just doesn't carry visual-spottability
signal for this task. Notebook 03's original feature set stays the one to
carry forward; this notebook's value is having actually tested the
alternative rather than assumed the answer.